#Preparación del entorno

**Proyecto:** Banking Analytics Lakehouse  
**Almacenamiento:** Azure Data Lake Storage  
**Catálogo:** `banking_dev`  
**Arquitectura:** Medallion 

Este notebook prepara el entorno de datos del proyecto, configurando los parámetros de almacenamiento, el catálogo de Unity Catalog, los esquemas y las tablas Delta externas necesarias para implementar la arquitectura Medallion.

Se definen las ubicaciones físicas de las tablas en ADLS, permitiendo mantener separados los datos de cada capa y administrar sus metadatos desde Unity Catalog.


In [0]:
%python
# CONFIGURACIÓN DEL ENTORNO
# Banking Analytics Lakehouse

# Widgets
dbutils.widgets.text("storage_account", "adlsprojectbank12")
dbutils.widgets.text("storage_credential", "credential-project")
dbutils.widgets.text("catalog_name", "banking_dev")

storage_account = dbutils.widgets.get("storage_account").strip()
storage_credential = dbutils.widgets.get("storage_credential").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

In [0]:
%python
print("Configuración del entorno")
print(f"Storage Account     : {storage_account}")
print(f"Storage Credential  : {storage_credential}")
print(f"Catálogo            : {catalog_name}")

Configuración del entorno
Storage Account     : adlsprojectbank12
Storage Credential  : credential-project
Catálogo            : banking_dev


In [0]:
%python
# DEFINICIÓN DE RUTAS 

# Contenedores
containers = ["raw", "bronze", "silver", "golden", "metastore"]

storage_paths = { 
    container: (f"abfss://{container}@{storage_account}.dfs.core.windows.net/") for container in containers
}

In [0]:
%python
for container, path in storage_paths.items():
    print(f"{container:<12}: {path}")

raw         : abfss://raw@adlsprojectbank12.dfs.core.windows.net/
bronze      : abfss://bronze@adlsprojectbank12.dfs.core.windows.net/
silver      : abfss://silver@adlsprojectbank12.dfs.core.windows.net/
golden      : abfss://golden@adlsprojectbank12.dfs.core.windows.net/
metastore   : abfss://metastore@adlsprojectbank12.dfs.core.windows.net/


In [0]:
%python
# STORAGE CREDENTIAL
credentials = spark.sql("SHOW STORAGE CREDENTIALS")

credential_exists = any(row["name"] == storage_credential
    for row in credentials.collect()
)

In [0]:
%python
if not credential_exists:
    raise Exception(
        f"No se ha encontrado ninguna Storage Credential: {storage_credential}"
    )

print(f"Storage Credential: {storage_credential}")

Storage Credential: credential-project


In [0]:
-- EXTERNAL LOCATIONS

-- Metastore
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-metastore`
URL 'abfss://metastore@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `${storage_credential}`)
COMMENT 'External Location para las tablas metastore del Data Lake';

-- Raw
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-raw`
URL 'abfss://raw@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `${storage_credential}`)
COMMENT 'External Location para las tablas raw del Data Lake';

-- Bronze
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-bronze`
URL 'abfss://bronze@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `${storage_credential}`)
COMMENT 'External Location para las tablas bronze del Data Lake';

-- Silver
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-silver`
URL 'abfss://silver@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `${storage_credential}`)
COMMENT 'External Location para las tablas silver del Data Lake';

-- Golden
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-banking-golden`
URL 'abfss://golden@${storage_account}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `${storage_credential}`)
COMMENT 'External Location para las tablas golden del Data Lake';


In [0]:
-- Validar External Locations
DESCRIBE EXTERNAL LOCATION `exlt-banking-metastore`;

name,url,credential_name,owner,created_by,created_at,comment
exlt-banking-metastore,abfss://metastore@adlsprojectbank12.dfs.core.windows.net/,credential-project,diego_smartdata_proy_12@outlook.com,diego_smartdata_proy_12@outlook.com,2026-11-10T03:36:30.803Z,External Location para las tablas metastore del Data Lake


In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-raw`;

name,url,credential_name,owner,created_by,created_at,comment
exlt-banking-raw,abfss://raw@adlsprojectbank12.dfs.core.windows.net/,credential-project,diego_smartdata_proy_12@outlook.com,diego_smartdata_proy_12@outlook.com,2026-11-10T03:37:22.806Z,External Location para las tablas raw del Data Lake


In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-bronze`;

name,url,credential_name,owner,created_by,created_at,comment
exlt-banking-bronze,abfss://bronze@adlsprojectbank12.dfs.core.windows.net/,credential-project,diego_smartdata_proy_12@outlook.com,diego_smartdata_proy_12@outlook.com,2026-11-10T03:39:07.318Z,External Location para las tablas bronze del Data Lake


In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-silver`;

name,url,credential_name,owner,created_by,created_at,comment
exlt-banking-silver,abfss://silver@adlsprojectbank12.dfs.core.windows.net/,credential-project,diego_smartdata_proy_12@outlook.com,diego_smartdata_proy_12@outlook.com,2026-11-10T03:39:13.468Z,External Location para las tablas silver del Data Lake


In [0]:
DESCRIBE EXTERNAL LOCATION `exlt-banking-golden`;

name,url,credential_name,owner,created_by,created_at,comment
exlt-banking-golden,abfss://golden@adlsprojectbank12.dfs.core.windows.net/,credential-project,diego_smartdata_proy_12@outlook.com,diego_smartdata_proy_12@outlook.com,2026-11-10T03:39:26.596Z,External Location para las tablas golden del Data Lake


In [0]:
-- CREAR EL CATÁLOGO
CREATE CATALOG IF NOT EXISTS ${catalog_name}
MANAGED LOCATION 'abfss://metastore@${storage_account}.dfs.core.windows.net/catalogs/${catalog_name}/';

In [0]:
-- CREAR SCHEMAS
CREATE SCHEMA IF NOT EXISTS ${catalog_name}.raw;

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.bronze;

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.silver;

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.golden;

##Creación de tablas Bronze

#### SBS

In [0]:
-- TABLAS BRONZE - SBS

CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.sbs_creditos_sector_economico (
    fecha_corte DATE,
    concepto STRING,
    nivel_jerarquia INT,
    numero_deudores BIGINT,
    creditos_moneda_nacional_miles_soles DECIMAL(18,2),
    creditos_moneda_extranjera_miles_soles DECIMAL(18,2),
    total_creditos_directos_miles_soles DECIMAL(18,2),
    distribucion_creditos_directos_pct DECIMAL(10,4),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_creditos_sector_economico';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.sbs_depositos_mn (
    fecha_corte DATE,
    moneda STRING,
    empresa STRING,
    numero_cuentas BIGINT,
    saldo_anterior_miles_soles DECIMAL(18,2),
    abonos_miles_soles DECIMAL(18,2),
    intereses_capitalizados_miles_soles DECIMAL(18,2),
    retiros_cargos_miles_soles DECIMAL(18,2),
    saldo_final_miles_soles DECIMAL(18,2),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_depositos_mn';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.sbs_depositos_me (
    fecha_corte DATE,
    moneda STRING,
    empresa STRING,
    numero_cuentas BIGINT,
    saldo_anterior_miles_soles DECIMAL(18,2),
    abonos_miles_soles DECIMAL(18,2),
    intereses_capitalizados_miles_soles DECIMAL(18,2),
    retiros_cargos_miles_soles DECIMAL(18,2),
    saldo_final_miles_soles DECIMAL(18,2),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_depositos_me';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.sbs_morosidad (
    fecha_corte DATE,
    concepto_credito STRING,
    nivel_jerarquia INT,
    empresa_bancaria STRING,
    morosidad_pct DECIMAL(10,4),
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/sbs_morosidad';

#### FRAUD

In [0]:
-- TABLA BRONZE - FRAUD
CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.fraud_transactions (
    Time DOUBLE,
    V1 DOUBLE,
    V2 DOUBLE,
    V3 DOUBLE,
    V4 DOUBLE,
    V5 DOUBLE,
    V6 DOUBLE,
    V7 DOUBLE,
    V8 DOUBLE,
    V9 DOUBLE,
    V10 DOUBLE,
    V11 DOUBLE,
    V12 DOUBLE,
    V13 DOUBLE,
    V14 DOUBLE,
    V15 DOUBLE,
    V16 DOUBLE,
    V17 DOUBLE,
    V18 DOUBLE,
    V19 DOUBLE,
    V20 DOUBLE,
    V21 DOUBLE,
    V22 DOUBLE,
    V23 DOUBLE,
    V24 DOUBLE,
    V25 DOUBLE,
    V26 DOUBLE,
    V27 DOUBLE,
    V28 DOUBLE,
    Amount DECIMAL(18,2),
    Class INT,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/fraud_transactions';

#### BANKING

In [0]:
-- TABLAS BRONZE - BANKING

CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.customers (
    customer_id BIGINT,
    document_type STRING,
    document_number STRING,
    first_name STRING,
    last_name STRING,
    birth_date DATE,
    gender STRING,
    city STRING,
    customer_segment STRING,
    registration_date DATE,
    customer_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/customers';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.accounts (
    account_id BIGINT,
    customer_id BIGINT,
    account_type STRING,
    currency STRING,
    opening_date DATE,
    balance DECIMAL(18,2),
    account_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/accounts';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.cards (
    card_id BIGINT,
    account_id BIGINT,
    card_type STRING,
    card_brand STRING,
    issue_date DATE,
    expiration_date DATE,
    card_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/cards';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.transactions (
    transaction_id BIGINT,
    account_id BIGINT,
    card_id BIGINT,
    transaction_date TIMESTAMP,
    transaction_type STRING,
    channel STRING,
    merchant_category STRING,
    amount DECIMAL(18,2),
    currency STRING,
    transaction_status STRING,
    is_fraud INT,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/transactions';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.loans (
    loan_id BIGINT,
    customer_id BIGINT,
    loan_type STRING,
    principal_amount DECIMAL(18,2),
    interest_rate DECIMAL(8,4),
    term_months INT,
    disbursement_date DATE,
    due_date DATE,
    loan_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/loans';


CREATE TABLE IF NOT EXISTS ${catalog_name}.bronze.payments (
    payment_id BIGINT,
    loan_id BIGINT,
    payment_date DATE,
    payment_amount DECIMAL(18,2),
    principal_paid DECIMAL(18,2),
    interest_paid DECIMAL(18,2),
    payment_status STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION 'abfss://bronze@${storage_account}.dfs.core.windows.net/payments';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.bronze.payments;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8a13159a-bc2f-4374-bc82-b54577373651,banking_dev.bronze.payments,null,abfss://bronze@adlsprojectbank12.dfs.core.windows.net/payments,2026-11-10T05:59:44.632Z,2026-11-14T17:11:43Z,List(),List(),1,272997,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.bronze;

database,tableName,isTemporary
bronze,accounts,false
bronze,cards,false
bronze,customers,false
bronze,fraud_transactions,false
bronze,loans,false
bronze,payments,false
bronze,sbs_creditos_sector_economico,false
bronze,sbs_depositos_me,false
bronze,sbs_depositos_mn,false
bronze,sbs_morosidad,false


##Creación de tablas Silver

In [0]:
-- TABLAS SILVER

--SBS
CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.sbs_creditos_sector_economico
LIKE ${catalog_name}.bronze.sbs_creditos_sector_economico
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_creditos_sector_economico';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.sbs_depositos_mn
LIKE ${catalog_name}.bronze.sbs_depositos_mn
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_depositos_mn';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.sbs_depositos_me
LIKE ${catalog_name}.bronze.sbs_depositos_me
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_depositos_me';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.sbs_morosidad
LIKE ${catalog_name}.bronze.sbs_morosidad
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/sbs_morosidad';

-- FRAUD
CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.fraud_transactions
LIKE ${catalog_name}.bronze.fraud_transactions
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/fraud_transactions';

-- BANKING
CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.customers
LIKE ${catalog_name}.bronze.customers
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/customers';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.accounts
LIKE ${catalog_name}.bronze.accounts
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/accounts';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.cards
LIKE ${catalog_name}.bronze.cards
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/cards';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.transactions
LIKE ${catalog_name}.bronze.transactions
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/transactions';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.loans
LIKE ${catalog_name}.bronze.loans
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/loans';

CREATE TABLE IF NOT EXISTS ${catalog_name}.silver.payments
LIKE ${catalog_name}.bronze.payments
LOCATION 'abfss://silver@${storage_account}.dfs.core.windows.net/payments';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.silver.payments;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,5bce1a1e-8e1d-4a01-9c7a-0771d9e0b6f4,banking_dev.silver.payments,null,abfss://silver@adlsprojectbank12.dfs.core.windows.net/payments,2026-11-10T06:03:07.354Z,2026-11-14T17:15:34Z,List(),List(),1,272997,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.silver;

database,tableName,isTemporary
silver,accounts,false
silver,cards,false
silver,customers,false
silver,fraud_transactions,false
silver,loans,false
silver,payments,false
silver,sbs_creditos_sector_economico,false
silver,sbs_depositos_me,false
silver,sbs_depositos_mn,false
silver,sbs_morosidad,false


## Creación de tablas Golden

In [0]:
-- TABLAS GOLDEN

CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.dim_customer (
    customer_key BIGINT,
    customer_id BIGINT,
    customer_segment STRING,
    city STRING,
    customer_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_customer';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.dim_account (
    account_key BIGINT,
    account_id BIGINT,
    customer_id BIGINT,
    account_type STRING,
    currency STRING,
    account_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_account';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.dim_card (
    card_key BIGINT,
    card_id BIGINT,
    account_id BIGINT,
    card_type STRING,
    card_brand STRING,
    card_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_card';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.dim_date (
    date_key INT,
    full_date DATE,
    year INT,
    quarter INT,
    month INT,
    month_name STRING,
    day INT,
    day_of_week INT
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/dim_date';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.fact_transactions (
    transaction_id BIGINT,
    date_key INT,
    customer_id BIGINT,
    account_id BIGINT,
    card_id BIGINT,
    amount DECIMAL(18,2),
    currency STRING,
    transaction_type STRING,
    channel STRING,
    transaction_status STRING,
    is_fraud INT
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_transactions';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.fact_loans (
    loan_id BIGINT,
    customer_id BIGINT,
    principal_amount DECIMAL(18,2),
    interest_rate DECIMAL(8,4),
    term_months INT,
    loan_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_loans';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.fact_payments (
    payment_id BIGINT,
    loan_id BIGINT,
    payment_date DATE,
    payment_amount DECIMAL(18,2),
    principal_paid DECIMAL(18,2),
    interest_paid DECIMAL(18,2),
    payment_status STRING
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/fact_payments';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.agg_sbs_creditos (
    fecha_corte DATE,
    concepto STRING,
    nivel_jerarquia INT,
    total_creditos_miles_soles DECIMAL(18,2),
    distribucion_pct DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_creditos';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.agg_sbs_depositos (
    fecha_corte DATE,
    empresa STRING,
    moneda STRING,
    numero_cuentas BIGINT,
    saldo_final_miles_soles DECIMAL(18,2)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_depositos';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.agg_sbs_morosidad (
    fecha_corte DATE,
    empresa_bancaria STRING,
    concepto_credito STRING,
    morosidad_pct DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_sbs_morosidad';


CREATE TABLE IF NOT EXISTS ${catalog_name}.golden.agg_fraud_daily (
    transaction_date DATE,
    total_transactions BIGINT,
    fraud_transactions BIGINT,
    legitimate_transactions BIGINT,
    fraud_amount DECIMAL(18,2),
    fraud_rate DECIMAL(10,4)
)
USING DELTA
LOCATION 'abfss://golden@${storage_account}.dfs.core.windows.net/agg_fraud_daily';

In [0]:
-- Validar ruta
DESCRIBE DETAIL banking_dev.golden.agg_fraud_daily;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,84428581-9681-4801-bdd9-267591a54f0e,banking_dev.golden.agg_fraud_daily,null,abfss://golden@adlsprojectbank12.dfs.core.windows.net/agg_fraud_daily,2026-11-10T06:23:12.336Z,2026-11-14T17:20:19Z,List(),List(),1,6763,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- Mostrar tablas en esquemas
SHOW TABLES IN banking_dev.golden;

database,tableName,isTemporary
golden,agg_fraud_daily,false
golden,agg_sbs_creditos,false
golden,agg_sbs_depositos,false
golden,agg_sbs_morosidad,false
golden,dim_account,false
golden,dim_card,false
golden,dim_customer,false
golden,dim_date,false
golden,fact_loans,false
golden,fact_payments,false


In [0]:
--Comprobar que todas las tablas sean external
SELECT
    table_schema,
    table_name,
    table_type
FROM banking_dev.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'golden')
ORDER BY table_schema, table_name;

table_schema,table_name,table_type
bronze,accounts,EXTERNAL
bronze,cards,EXTERNAL
bronze,customers,EXTERNAL
bronze,fraud_transactions,EXTERNAL
bronze,loans,EXTERNAL
bronze,payments,EXTERNAL
bronze,sbs_creditos_sector_economico,EXTERNAL
bronze,sbs_depositos_me,EXTERNAL
bronze,sbs_depositos_mn,EXTERNAL
bronze,sbs_morosidad,EXTERNAL


In [0]:
select * from banking_dev.golden.agg_fraud_daily
limit 100;

transaction_date,total_transactions,fraud_transactions,legitimate_transactions,fraud_amount,fraud_rate
2025-02-16,1398,1,1397,996.27,0.0715
2025-06-29,1386,1,1385,18.00,0.0722
2025-06-08,1329,2,1327,355.33,0.1505
2025-02-01,1385,4,1381,157.18,0.2888
2025-02-15,1449,8,1441,969.40,0.5521
2025-03-23,1388,2,1386,10.71,0.1441
2025-04-17,1365,2,1363,192.14,0.1465
2025-02-05,1345,2,1343,469.39,0.1487
2025-05-13,1387,0,1387,0.00,0.0000
2025-02-13,1429,3,1426,100.77,0.2099
